In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
sys.path.insert(0, project_root)

In [ ]:
import logging
import os

from dotenv import load_dotenv
from opensearchpy import OpenSearch

from src.retrieval.embedder import Embedder
from src.retrieval.searcher import HybridSearcher


In [ ]:
load_dotenv()

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [ ]:
client = OpenSearch(
    hosts=[{
        "host": os.getenv("OPENSEARCH_HOST", "localhost"),
        "port": int(os.getenv("OPENSEARCH_PORT", 9200))
    }],
    http_compress=True
)

embedder = Embedder(ollama_url=os.getenv("OLLAMA_URL"))
searcher = HybridSearcher(client=client, embedder=embedder, top_k=5)

In [ ]:
query = "Qual o tratamento para crise aguda de angioedema hereditário?"
results = searcher.search(query)

print(f"\nQuery: {query}")
print(f"{'='*60}")
for i, result in enumerate(results):
    print(f"\n[{i+1}] Score: {result.score:.4f}")
    print(f"     Source: {result.source}")
    print(f"     Tokens: {result.metadata['token_count']}")
    print(f"     Text: {result.text[:200]}")